In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o")

small_llm = ChatOpenAI(model="gpt-4o-mini")

In [3]:
from langchain_core.tools import tool


@tool
def add(a: int, b: int) -> int:
    """숫자 a와 b를 더합니다."""
    return a + b


@tool
def multiply(a: int, b: int) -> int:
    """숫자 a와 b를 곱합니다."""
    return a * b

In [4]:
# tool을 llm이 쓸 수 있도록 등록하고 할당
llm_with_tools = small_llm.bind_tools([add, multiply])

In [5]:
query = "3 곱하기 5는?"

In [6]:
# 그냥 llm을 호출을 하면 -> AIMesasge의 content에 답이 담긴다.
small_llm.invoke(query)

AIMessage(content='3 곱하기 5는 15입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 15, 'total_tokens': 26, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_046890a2c5', 'id': 'chatcmpl-DlsIApBKUAoAZM7yfVBlp8kV4PgU6', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e824c-1498-7e70-8c2f-9ceff02a36bd-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 15, 'output_tokens': 11, 'total_tokens': 26, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [7]:
# tool이 bind된 llm을 호출하면 -> AIMessage의 content는 비어있고, tool_calls에 호출 요청이 담긴다.
# llm_with_tools.invoke(query)
result = llm_with_tools.invoke(query)

In [8]:
# tool_calls에 호출할 tool들이 list로 담긴다.
result.tool_calls

[{'name': 'multiply',
  'args': {'a': 3, 'b': 5},
  'id': 'call_qythCNuCoLjhzjSZWeOI1ext',
  'type': 'tool_call'}]

LLM이 tool을 invoke 할 수 있도록(사용할 수 있도록) 세팅

In [9]:
from typing import Sequence

from langchain_core.messages import AnyMessage, HumanMessage

human_message = HumanMessage(query)
message_list: Sequence[AnyMessage] = [human_message]

In [10]:
# message_list에는 human_message만 담겨있는 상태.
ai_message = llm_with_tools.invoke(message_list)
# tool이 binding 되어있는 llm_with_tools가 message_list를 invoke 했을 때 -> text 답변 대신 tool_calls에 요청을 담아서 돌려준다.
# "내가 직접 답하는 것보다 도구를 쓰는게 낫겠다" 라고 LLM이 판단
# ai_message에는 LLM의 응답(tool_calls 요청이 담긴다)

In [11]:
# ai_message를 보면 text는 없고, tool_calls만 다음과 같이 반환해준다.
ai_message.tool_calls
# LLM이 개발자에게 "이 질문은 multiply라는 tool에 a=3, b=5로 호출하면 답을 받을 수 있어요" 라고 추천하는 것.

[{'name': 'multiply',
  'args': {'a': 3, 'b': 5},
  'id': 'call_H8oOZ4xmysjDW7S6EgNV4QGD',
  'type': 'tool_call'}]

In [12]:
# message_list에 추가적으로 ai_message와 tool_message를 append 해줘야한다.
# 일단 ai_message 추가
message_list.append(ai_message)

In [13]:
# 도구의 호출을 ai_message로 한다.
# multiply.invoke(ai_message.tool_calls[0])
# -> ToolMessage가 반환된다. 이 ToolMessage도 message_list에 append한다.
tool_message = multiply.invoke(ai_message.tool_calls[0])

In [14]:
message_list.append(tool_message)

# 결과적으로 message_list에는
# HumanMessage: "3 곱하기 5는?"
# AIMessage: "multiply(a=3, b=5) 호출해줘 (tool_calls)"
# ToolMessage: "결과: 15 (도구 실행 결과)"
# 가 담겨있다.

In [15]:
llm_with_tools.invoke(message_list)

AIMessage(content='3 곱하기 5는 15입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 12, 'prompt_tokens': 109, 'total_tokens': 121, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_046890a2c5', 'id': 'chatcmpl-DlsIDdtrD94ePqPnEX9uB0QMVma4o', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e824c-2656-7572-b550-c6a966579f1c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 109, 'output_tokens': 12, 'total_tokens': 121, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})